In [174]:
import warnings
warnings.filterwarnings('ignore')

import os
import gc
import pickle

import numpy as np
import pandas as pd
# import matplotlib.pyplot as plt

from datetime import datetime
from pandas.tseries.offsets import MonthEnd

In [175]:
# define all variables
base_dir = '/data/aman_singh/acuuracy_check'
last_month = '2026-02-28'
run_month = '2026-03-31'
channel = 'GT'
collated_path = "/data/aman_singh/acuuracy_check/all_channels brand_asm mar'26.csv"

In [176]:
def list_all_files_in_directory(root):
    out = []

    for path, subdirs, files in os.walk(root):
        for name in files:
            out.append(os.path.join(path, name))

    return out

In [177]:
def discover_channel(file_path):
    file_path = file_path.split('\\')[2]

    if 'ECOM' in file_path:
        return 'ECOM'
    elif 'QCOM' in file_path:
        return 'QCOM'
    elif 'MT' in file_path:
        return 'MT'
    elif 'GT' in file_path:
        return 'GT'
    else:
        return 'Channel not found'

    # return "MT"


In [178]:
def get_run_month(train_till_str):
    return datetime.strptime(train_till_str, 'train_till_%d_%b_%Y') + MonthEnd(1)

In [179]:
from maricovault.MaricoDB import MaricoSnowflake

def get_dbconnection(db_name):    

    KEY_VAULT_NAME = "prod-pwd"

    if db_name == 'PROD':
        db_name = 'prod'
    else:
        db_name = 'dev'
    

    msf = MaricoSnowflake(KEY_VAULT_NAME)
    msf.get_db_credentials(db_name=db_name)
    msf.connect()
    dbconnection = msf.get_connection()
    
    return dbconnection


def read_qtr_ind_rate_table():
    """
    Fetch the club sku information from  DWH_SAP_INDEX_TURNOVER_MONTHWISE table.

    Return:
        qtr_ind_rate_data: pandas dataframe
        - dataframe contains all the results from the index rate table.
    """
    connection = get_dbconnection(db_name='PROD')
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=connection, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    connection.close()
    return qtr_ind_rate


dev_conn = get_dbconnection('DEV')
prod_conn = get_dbconnection('PROD')


Credentials retrieved successfully for dev db.

Credentials retrieved successfully for prod db.


In [180]:
# query = f""" 
# SELECT 
#     channel_name,
#     asm_area_code, 
#     depot_code, 
#     parent_material1_code, 
#     month_date, 
#     sum(sec_actuals_vol_rum_month) Sec_Vol_Actuals_Rum_Month,
#     sum(sec_apo_plan_vol_rum_month) Sec_Vol_Apo_Plan_Rum_Month
# FROM (
#     Select 
#         Month_Date, 
#         Distributor_Code, 
#         material_code, 
#         sec_actuals_vol_rum_month, 
#         sec_apo_plan_vol_rum_month
#     from 
#         dwh_bpm_dist_brand_mth_sbp 
#     where 
#     month_date  between '2022-01-01' AND '{last_month}') A

# JOIN (
#     SELECT
#         channel_name, 
#         customer_code, 
#         asm_area_code, 
#         depot_code
#     FROM 
#         mst_customer 
#     WHERE 
#         company_code= 'MIL' 
#         and latest_record_ind=1 
#         and channel_name in ('MT', 'E-Commerce', 'Q-Commerce', 'GT')) C 
#     ON 
#         distributor_code = customer_code
# JOIN (
#     Select 
#         material_code, 
#         parent_material1_code 
#     from 
#         mst_material 
#     where 
#         company_code= 'MIL' 
#         and latest_record_ind=1) M 
#     ON 
#         A.material_code = M.material_code
#     group by 
#         channel_name, asm_area_code, depot_code, parent_material1_code, Month_date
#     ORDER BY 
#         channel_name, asm_area_code, depot_code, parent_material1_code, Month_date
# """
# # GT
# results = pd.read_sql(con=prod_conn, sql=query)
# sales_data = pd.DataFrame(results)
# sales_data.columns = sales_data.columns.str.lower()
# sales_data = sales_data.rename(columns={'parent_material1_code':'parent_material_code'})

In [181]:
# extract data for all channels
query = f"""SELECT
    CM.channel_name,
    CM.asm_area_code,
    MM.material_group_code as brand_code,
    LAST_DAY(MESR.month_date) AS month_date,
    SUM(MESR.sec_apo_plan_vol_rum) AS Sec_Vol_Apo_Plan_Rum_Month,
    SUM(MESR.sec_actuals_vol_rum) AS Sec_Vol_Actuals_Rum_Month
FROM
    dwh_bpm_dist_sku_daily MESR
JOIN
(
    SELECT
        material_code,
        parent_material_code,
        material_group_code,
        uom_reporting,
        vol_per_unit
    FROM 
        mst_material
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
        
) MM ON MESR.material_code = MM.material_code
JOIN
(
    SELECT DISTINCT
        channel_name, 
        asm_area_code,
        customer_code,
        depot_code
    FROM
        mst_customer
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
        AND channel_name in ('E-Commerce', 'GT', 'MT', 'Q-Commerce') 
) CM ON MESR.distributor_code = CM.customer_code
WHERE
    month_date BETWEEN '2023-01-01' and '{last_month}'
    
GROUP BY 1, 2, 3, 4
ORDER BY 1, 2, 3, 4
"""

results = pd.read_sql(con=prod_conn, sql=query)
sales_data = pd.DataFrame(results)
sales_data.columns = sales_data.columns.str.lower()
#sales_data = sales_data.rename(columns={'parent_material1_code':'parent_material_code'})
sales_data

,channel_name,asm_area_code,brand_code,month_date,sec_vol_apo_plan_rum_month,sec_vol_actuals_rum_month
0,E-Commerce,BCE1,ADV-AHO-R,2023-01-31,2.2900,33.60
1,E-Commerce,BCE1,ADV-AHO-R,2023-02-28,4.6950,30.30
2,E-Commerce,BCE1,ADV-AHO-R,2023-03-31,2.2240,18.00
3,E-Commerce,BCE1,ADV-AHO-R,2023-04-30,22.9140,5.10
4,E-Commerce,BCE1,ADV-AHO-R,2023-05-31,31.8380,12.90
...,...,...,...,...,...,...
251919,Q-Commerce,QCW2,SW_SGPRF,2025-10-31,2.3902,0.00
251920,Q-Commerce,QCW2,SW_SGPRF,2025-11-30,7.0400,0.00
251921,Q-Commerce,QCW2,SW_SGPRF,2025-12-31,0.4700,20.88
251922,Q-Commerce,QCW2,SW_SGPRF,2026-01-31,0.7600,0.00


In [182]:
realignment_df = pd.read_sql(
    'select * from trn_mil_asm_psku_realignment',
    dev_conn
)
realignment_df.columns = realignment_df.columns.str.lower()

def realign_pskus(data, channel, columns=['sec_vol_actuals_rum_month', 'pri_actuals_vol_rum_month']):

    realignment_data = realignment_df.copy()
    assert realignment_data['psku old'].dtype == 'int64'
    assert realignment_data['psku new'].dtype == 'int64'
    
    realignment_data = realignment_data[
        (realignment_data["channel"] == channel)
        | (realignment_data["channel"] == channel + " B2C")
        | (realignment_data["channel"] == "ALL")
    ]

    data["parent_material_code"] = data["parent_material_code"].astype(int)

    for grp, grp_data in realignment_data.groupby(by=["psku old", "asm"]):
        old_psku, old_asm = grp
        new_psku = grp_data["psku new"].values[0]
        if old_asm != "ALL":
            condition = (data["parent_material_code"] == old_psku) & (
                data["asm_area_code"] == old_asm
            )
        else:
            condition = data["parent_material_code"] == old_psku

        data.loc[condition, "parent_material_code"] = new_psku


    data = data.groupby(
        ["channel_name", "asm_area_code", 'depot_code', "parent_material_code", "month_date"],
        as_index=False,
    )[columns].agg('sum')

    return data

In [183]:
sales_data['month_date'].max()

datetime.date(2026, 2, 28)

In [184]:
sales_data['channel_name'].replace({'E-Commerce': 'ECOM', 'Q-Commerce': 'QCOM'}, inplace=True)

In [185]:
sales_data['channel_name'].unique()

array(['ECOM', 'GT', 'MT', 'QCOM'], dtype=object)

In [186]:
# realigned_df = pd.DataFrame()

# for channel in sales_data['channel_name'].unique():
#     tmp_df = sales_data[sales_data['channel_name'] == channel]
#     tmp_df = realign_pskus(tmp_df, channel=channel, columns=['sec_vol_actuals_rum_month'])
#     realigned_df = pd.concat([realigned_df, tmp_df])
#     del tmp_df

In [187]:
# realigned_df.duplicated(
#     subset=['channel_name', 'asm_area_code', 'depot_code', 
#             'parent_material_code', 'month_date']
# ).sum()

In [188]:
sales_data = sales_data[sales_data['asm_area_code'].notna()]
sales_data.isnull().sum()

channel_name                  0
asm_area_code                 0
brand_code                    0
month_date                    0
sec_vol_apo_plan_rum_month    0
sec_vol_actuals_rum_month     0
dtype: int64

In [189]:
realigned_df = sales_data.copy()
realigned_df['key'] = realigned_df['asm_area_code'] + '_' + realigned_df['brand_code']  
realigned_df['month_date'] = pd.to_datetime(realigned_df['month_date'])

realigned_df.duplicated(subset=['channel_name', 'key', 'month_date']).sum()

0

In [190]:
realigned_df = realigned_df.groupby(
    ['channel_name', 'key', 'asm_area_code', 'brand_code', 'month_date'],
    as_index=False
)['sec_vol_actuals_rum_month'].sum()

In [191]:
# realigned_df.to_csv('OT_data_debug.csv', index=False)

### Collate MIL

In [192]:
def collate_file(file_hint, extension='.csv'):
    collated_file = pd.DataFrame()

    for run in mil_runs:
        run_path = f'{base_dir}\\{run}'
        all_files = list_all_files_in_directory(run_path)

        for file_path in all_files:
            if file_hint in file_path:
                if extension == '.csv':
                    print(file_path)
                    read_file = pd.read_csv(file_path)
                    read_file['channel'] = discover_channel(file_path)
                    read_file['run_month'] = get_run_month(file_path.split('\\')[4])
                    read_file['run'] = run
                    read_file['step'] = file_path.split('\\')[3]
                    read_file['file_path'] = file_path

                    collated_file = pd.concat(
                        [collated_file, read_file]
                    )
                    del read_file

    return collated_file

In [193]:
# trend_file_df = collate_file('trend_file_train_till')
# prophet_file_df = collate_file('prophet_data_train_till')
trend_file_df = pd.read_sql(
    f"""
    select * from TRN_MIL_DF_ASM_BRAND
    where run_month='{run_month}'
""",
dev_conn
)
trend_file_df.columns = trend_file_df.columns.str.lower()

In [194]:
trend_file_df.head()

,channel,asm_area_code,brand_code,run_month,month_date,sec_vol_actuals_rum_month,extra_vol_co,other_co,price_off_co,seasonal_month_flag,diwali,ganesh_chaturthi,drive,outlier
0,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-04-30,18.0,0.0,0.0,0.0,0,0,0,NaN,NaN
1,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-05-31,9.0,0.0,0.0,0.0,0,0,0,NaN,NaN
2,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-06-30,72.0,0.0,0.0,0.0,0,0,0,NaN,NaN
3,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-07-31,0.0,0.0,0.0,0.0,0,0,0,NaN,NaN
4,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-08-31,0.0,0.0,0.0,0.0,0,0,1,NaN,NaN


In [195]:
trend_file_df['channel'].unique()

array(['ECOM', 'GT', 'MT', 'QCOM'], dtype=object)

In [196]:
trend_file_df.columns

Index(['channel', 'asm_area_code', 'brand_code', 'run_month', 'month_date',
       'sec_vol_actuals_rum_month', 'extra_vol_co', 'other_co', 'price_off_co',
       'seasonal_month_flag', 'diwali', 'ganesh_chaturthi', 'drive',
       'outlier'],
      dtype='object')

In [197]:
trend_file_df['month_date'] = pd.to_datetime(trend_file_df['month_date'])
trend_file_df['run_month'] = pd.to_datetime(trend_file_df['run_month'])


In [198]:
# data_file_df['run_month'] = pd.to_datetime(data_file_df['run_month'])

In [199]:
# data_file_df.columns

In [200]:
# missing_forecasts = pd.DataFrame()

# for rm_dt in trend_file_df['run_month'].unique():
#     for c in trend_file_df['channel'].unique():
#         tmp_df = data_file_df[
#             (data_file_df['run_month'] == rm_dt) &
#             (data_file_df['channel'] == c) &
#             (~data_file_df['key'].isin(trend_file_df[
#                 (trend_file_df['run_month'] == rm_dt) &
#                 (trend_file_df['channel'] == c)
#             ]['key'].unique()))
#         ]

#         missing_forecasts = pd.concat([missing_forecasts, tmp_df]).reset_index(drop=True)
    

In [201]:
# data_file_df.duplicated(
#     subset=['channel', 'run_month', 'month_date', 'asm_area_code',
#             'depot_code', 'parent_material_code']
# ).sum()

In [202]:
# missing_forecasts['tmp_key'] = missing_forecasts[['channel', 'key', 'run_month']].astype(str).agg('_'.join, axis=1)

In [203]:
# tmp_trend_df = trend_file_df.copy()
# tmp_trend_df['tmp_key'] = trend_file_df[['channel', 'key', 'run_month']].astype(str).agg('_'.join, axis=1)

# set(missing_forecasts['tmp_key'].unique()).intersection(tmp_trend_df['tmp_key'].unique())

In [204]:
# del tmp_trend_df, missing_forecasts['tmp_key']

In [205]:
trend_file_df.columns

Index(['channel', 'asm_area_code', 'brand_code', 'run_month', 'month_date',
       'sec_vol_actuals_rum_month', 'extra_vol_co', 'other_co', 'price_off_co',
       'seasonal_month_flag', 'diwali', 'ganesh_chaturthi', 'drive',
       'outlier'],
      dtype='object')

In [206]:
trend_file_df['channel'].unique() #, prophet_file_df['channel'].unique()

array(['ECOM', 'GT', 'MT', 'QCOM'], dtype=object)

In [207]:
for channel in trend_file_df['channel'].unique():
    print(trend_file_df[trend_file_df['channel'] == channel]['asm_area_code'].unique())

['BCE1' 'BCE2' 'BCN1' 'BCN2' 'BCS1' 'BCS2' 'BCW1' 'BCW2' 'ECE1' 'ECE2'
 'ECN1' 'ECN2' 'ECS1' 'ECS2' 'ECW1' 'ECW2']
['AURG' 'BIHE' 'BIHW' 'BLR' 'CMB' 'CNI' 'CTGH' 'CUP' 'DELM' 'EMP' 'EUP'
 'GUJN' 'GUJS' 'HAR' 'HYD1' 'HYD2' 'JHR' 'KARC' 'KARN' 'KOL' 'KRL' 'MST'
 'MUM1' 'MUM2' 'NAG' 'NERE' 'NERW' 'NUP' 'ORS' 'PJB' 'PUNN' 'PUNS' 'RAJ1'
 'RAJ2' 'RWBN' 'RWBS' 'TPT' 'VIJ' 'WMP' 'WUP']
['BCE1' 'BCE2' 'BCN1' 'BCN2' 'BCS1' 'BCS2' 'BCW1' 'BCW2' 'MCE1' 'MCE2'
 'MCN1' 'MCN2' 'MCS1' 'MCS2' 'MCW1' 'MCW2']
['QCE1' 'QCE2' 'QCN1' 'QCN2' 'QCS1' 'QCS2' 'QCW1' 'QCW2']


In [208]:
trend_file_df

,channel,asm_area_code,brand_code,run_month,month_date,sec_vol_actuals_rum_month,extra_vol_co,other_co,price_off_co,seasonal_month_flag,diwali,ganesh_chaturthi,drive,outlier
0,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-04-30,18.0,0.0,0.0,0.0,0,0,0,NaN,NaN
1,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-05-31,9.0,0.0,0.0,0.0,0,0,0,NaN,NaN
2,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-06-30,72.0,0.0,0.0,0.0,0,0,0,NaN,NaN
3,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-07-31,0.0,0.0,0.0,0.0,0,0,0,NaN,NaN
4,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-08-31,0.0,0.0,0.0,0.0,0,0,1,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
214622,QCOM,QCW2,SW_SGPRF,2026-03-31,2026-11-30,0.0,0.0,0.0,0.0,0,1,0,NaN,NaN
214623,QCOM,QCW2,SW_SGPRF,2026-03-31,2026-12-31,0.0,0.0,0.0,0.0,0,0,0,NaN,NaN
214624,QCOM,QCW2,SW_SGPRF,2026-03-31,2027-01-31,0.0,0.0,0.0,0.0,0,0,0,NaN,NaN
214625,QCOM,QCW2,SW_SGPRF,2026-03-31,2027-02-28,0.0,0.0,0.0,0.0,0,0,0,NaN,NaN


In [209]:
trend_file_df['key'] = trend_file_df[['asm_area_code', 'brand_code']].astype(str).agg('_'.join, axis=1)

In [210]:
trend_file_df.duplicated(subset=['key', 'channel', 'month_date', 'run_month']).sum()

0

In [211]:
trend_file_df = trend_file_df.drop_duplicates(subset=['key', 'channel', 'month_date', 'run_month'])

In [212]:
trend_file_df.duplicated(subset=['key', 'channel', 'month_date', 'run_month']).sum()

0

In [213]:
mappings = {}

for run_month in trend_file_df['run_month'].unique():
    # run_month = pd.to_datetime(run_month)

    if not run_month in mappings:
        mappings[run_month] = {}

    mappings[run_month][run_month] = 'M'

    for idx in range(1, 13):
        mappings[run_month][run_month + MonthEnd(idx)] = f"M+{idx}"


mappings   

{Timestamp('2026-03-31 00:00:00'): {Timestamp('2026-03-31 00:00:00'): 'M',
  Timestamp('2026-04-30 00:00:00'): 'M+1',
  Timestamp('2026-05-31 00:00:00'): 'M+2',
  Timestamp('2026-06-30 00:00:00'): 'M+3',
  Timestamp('2026-07-31 00:00:00'): 'M+4',
  Timestamp('2026-08-31 00:00:00'): 'M+5',
  Timestamp('2026-09-30 00:00:00'): 'M+6',
  Timestamp('2026-10-31 00:00:00'): 'M+7',
  Timestamp('2026-11-30 00:00:00'): 'M+8',
  Timestamp('2026-12-31 00:00:00'): 'M+9',
  Timestamp('2027-01-31 00:00:00'): 'M+10',
  Timestamp('2027-02-28 00:00:00'): 'M+11',
  Timestamp('2027-03-31 00:00:00'): 'M+12'}}

In [214]:
trend_file_df['M month'] = trend_file_df.apply(
    lambda x: mappings[x['run_month']].get(
        x['month_date']
    ), axis=1
)

In [215]:
# missing_forecasts['M month'] = missing_forecasts.apply(
#     lambda x: mappings[x['run_month']].get(
#         x['month_date']
#     ), axis=1
# )

In [216]:
# trend_file_df[['run_month', 'train_till']].drop_duplicates().sort_values(by=['run_month'])

In [217]:
# prophet_file_df[['run_month', 'train_till']].drop_duplicates().sort_values(by=['run_month'])

In [218]:
trend_file_df['M month'].unique()

array([None, 'M', 'M+1', 'M+2', 'M+3', 'M+4', 'M+5', 'M+6', 'M+7', 'M+8',
       'M+9', 'M+10', 'M+11', 'M+12'], dtype=object)

In [219]:
trend_file_df

,channel,asm_area_code,brand_code,run_month,month_date,sec_vol_actuals_rum_month,extra_vol_co,other_co,price_off_co,seasonal_month_flag,diwali,ganesh_chaturthi,drive,outlier,key,M month
0,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-04-30,18.0,0.0,0.0,0.0,0,0,0,NaN,NaN,BCE1_ADV-AHO-R,None
1,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-05-31,9.0,0.0,0.0,0.0,0,0,0,NaN,NaN,BCE1_ADV-AHO-R,None
2,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-06-30,72.0,0.0,0.0,0.0,0,0,0,NaN,NaN,BCE1_ADV-AHO-R,None
3,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-07-31,0.0,0.0,0.0,0.0,0,0,0,NaN,NaN,BCE1_ADV-AHO-R,None
4,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-08-31,0.0,0.0,0.0,0.0,0,0,1,NaN,NaN,BCE1_ADV-AHO-R,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
214622,QCOM,QCW2,SW_SGPRF,2026-03-31,2026-11-30,0.0,0.0,0.0,0.0,0,1,0,NaN,NaN,QCW2_SW_SGPRF,M+8
214623,QCOM,QCW2,SW_SGPRF,2026-03-31,2026-12-31,0.0,0.0,0.0,0.0,0,0,0,NaN,NaN,QCW2_SW_SGPRF,M+9
214624,QCOM,QCW2,SW_SGPRF,2026-03-31,2027-01-31,0.0,0.0,0.0,0.0,0,0,0,NaN,NaN,QCW2_SW_SGPRF,M+10
214625,QCOM,QCW2,SW_SGPRF,2026-03-31,2027-02-28,0.0,0.0,0.0,0.0,0,0,0,NaN,NaN,QCW2_SW_SGPRF,M+11


In [220]:
trend_file_df.columns

Index(['channel', 'asm_area_code', 'brand_code', 'run_month', 'month_date',
       'sec_vol_actuals_rum_month', 'extra_vol_co', 'other_co', 'price_off_co',
       'seasonal_month_flag', 'diwali', 'ganesh_chaturthi', 'drive', 'outlier',
       'key', 'M month'],
      dtype='object')

In [221]:
trend_file_df.duplicated(subset=['key', 'channel', 'month_date', 'run_month']).sum()

0

In [222]:
brand_md_df = pd.read_excel(r"/data/aman_singh/mt_forecast/Brand_metadata.xlsx")

In [223]:
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    brand_md_df,
    on=['brand_code'],
    how='left'
)
assert len_before_merge == len(trend_file_df)

In [224]:
qtr_ind_df = read_qtr_ind_rate_table()
qtr_ind_df.head()


Credentials retrieved successfully for prod db.


,month_date,brand_code,qtr_ind_rate
0,2026-03-31,PADV_WIPS,168.752000
1,2026-03-31,PABABY_SP,415.245000
2,2026-03-31,P_GOHR_SR,7470.000000
3,2026-03-31,PA_JAS_GD,298.847906
4,2026-03-31,PA_NOR_HO,188992.000000


In [225]:
trend_file_df['portfolio'].isna().sum()

196

In [226]:
trend_file_df.columns

Index(['channel', 'asm_area_code', 'brand_code', 'run_month', 'month_date',
       'sec_vol_actuals_rum_month', 'extra_vol_co', 'other_co', 'price_off_co',
       'seasonal_month_flag', 'diwali', 'ganesh_chaturthi', 'drive', 'outlier',
       'key', 'M month', 'portfolio'],
      dtype='object')

In [227]:
assert qtr_ind_df.duplicated(subset=['brand_code']).sum() == 0
qtr_ind_df.drop('month_date', axis=1, inplace=True)

In [228]:
#qtr_ind_df.drop('qtr_ind_rate', axis=1, inplace=True)

In [229]:
qtr_ind_df

,brand_code,qtr_ind_rate
0,PADV_WIPS,168.752000
1,PABABY_SP,415.245000
2,P_GOHR_SR,7470.000000
3,PA_JAS_GD,298.847906
4,PA_NOR_HO,188992.000000
...,...,...
337,LIVON HG Shampoo,0.000000
338,NHO ACE,0.000000
339,SMO Premix,197307.938220
340,LVN_CONDI,552.140000


In [230]:
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    qtr_ind_df,
    on=['brand_code'],
    how='left'
)
assert len_before_merge == len(trend_file_df)

In [231]:
assert trend_file_df.duplicated(
    subset=['run_month', 'month_date', 'channel', 'key']
).sum() == 0

In [232]:
trend_file_df.drop('sec_vol_actuals_rum_month', axis=1, inplace=True)

In [233]:
realigned_df.dtypes

channel_name                         object
key                                  object
asm_area_code                        object
brand_code                           object
month_date                   datetime64[ns]
sec_vol_actuals_rum_month           float64
dtype: object

In [234]:
assert realigned_df.duplicated(subset=['channel_name', 'key', 'month_date']).sum() == 0

In [235]:
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    realigned_df[['channel_name', 'key', 'month_date', 'sec_vol_actuals_rum_month']].rename(
        columns={
            'channel_name': 'channel'
        }
    ),
    on=['month_date', 'channel', 'key'],
    how='left'
)
assert len(trend_file_df) == len_before_merge

In [236]:
trend_file_df.select_dtypes('number').isna().sum()

extra_vol_co                      0
other_co                          0
price_off_co                      0
seasonal_month_flag               0
diwali                            0
ganesh_chaturthi                  0
drive                        104955
outlier                      104955
qtr_ind_rate                     98
sec_vol_actuals_rum_month     88626
dtype: int64

In [237]:
trend_file_df.select_dtypes('number').min().round()

extra_vol_co                     0.0
other_co                         0.0
price_off_co                     0.0
seasonal_month_flag              0.0
diwali                           0.0
ganesh_chaturthi                 0.0
drive                            0.0
outlier                          0.0
qtr_ind_rate                     0.0
sec_vol_actuals_rum_month   -20180.0
dtype: float64

In [238]:
# 'pred_prophet_70%ile',
for col in ['sec_vol_actuals_rum_month']:
    trend_file_df[col] = trend_file_df[col].clip(lower=0)

In [239]:
trend_file_df = trend_file_df.sort_values(
    ['run_month', 'brand_code', 'channel', 'key', 'month_date']
)

In [240]:
trend_file_df = trend_file_df.fillna(0)

In [241]:
trend_file_df['P3M'] = trend_file_df.groupby(['run_month', 'channel', 'key'], as_index = False, group_keys = False)['sec_vol_actuals_rum_month'].shift(1)\
                                .rolling(window=3, min_periods=1).mean()

trend_file_df['P6M'] = trend_file_df.groupby(['run_month', 'channel', 'key'], as_index = False, group_keys = False)['sec_vol_actuals_rum_month'].shift(1)\
                                .rolling(window=6, min_periods=6).mean()

trend_file_df['LY P3M'] = trend_file_df.groupby(['run_month', 'channel', 'key'], as_index = False, group_keys = False)['sec_vol_actuals_rum_month'].shift(13)\
                                .rolling(window=3, min_periods=3).mean()

trend_file_df['LY P6M'] = trend_file_df.groupby(['run_month', 'channel', 'key'], as_index = False, group_keys = False)['sec_vol_actuals_rum_month'].shift(13)\
                                .rolling(window=6, min_periods=6).mean()

In [242]:
trend_file_df['LY P3M_copy'] = trend_file_df['LY P3M'].copy()

In [243]:
# pd.Series([100, 2, 4, 6]).nlargest(2).mean()

In [244]:
np.sort(np.array([100, 2, 4, 6]))[-2:].mean()

53.0

In [245]:
trend_file_df.columns

Index(['channel', 'asm_area_code', 'brand_code', 'run_month', 'month_date',
       'extra_vol_co', 'other_co', 'price_off_co', 'seasonal_month_flag',
       'diwali', 'ganesh_chaturthi', 'drive', 'outlier', 'key', 'M month',
       'portfolio', 'qtr_ind_rate', 'sec_vol_actuals_rum_month', 'P3M', 'P6M',
       'LY P3M', 'LY P6M', 'LY P3M_copy'],
      dtype='object')

In [246]:
trend_file_df['P3M Max'] = trend_file_df.groupby(['run_month', 'channel', 'key'], as_index = False, group_keys = False)['P3M'].shift(1)\
                                .rolling(window=3, min_periods=3).max()

In [247]:
trend_file_df['P3M Top 2 Mean'] = trend_file_df.groupby(['run_month', 'channel', 'key'], as_index = False, group_keys = False)['P3M'].shift(1)\
                                .rolling(window=3, min_periods=3).apply(lambda x: np.sort(x)[-2:].mean(), raw=True) 

# lambda x: x.nlargest(2).mean(), raw=False

In [248]:
trend_file_df = trend_file_df.sort_values(
    ['run_month', 'brand_code', 'channel', 'key', 'month_date']
)

In [249]:
trend_file_df['MoM P3M growth'] = (
    trend_file_df.groupby(['run_month', 'channel', 'key'])['P3M']
      .pct_change() * 100
)

In [250]:
(np.array([40, 10, 20]) > 20).sum()

1

In [251]:
np.sum(np.array([40, 10, 20]) > 20)

1

In [252]:
trend_file_df = trend_file_df.sort_values(
    ['channel', 'run_month', 'brand_code', 'key', 'month_date']
)
trend_file_df['MoM P3M growth_lag_1'] = trend_file_df.groupby(
    ['channel', 'run_month', 'brand_code', 'key']
)['MoM P3M growth'].shift(1)
trend_file_df['MoM P3M growth_lag_2'] = trend_file_df.groupby(
    ['channel', 'run_month', 'brand_code', 'key']
)['MoM P3M growth'].shift(2)



In [253]:
trend_file_df['>=20%_3M_inc_month_count'] = trend_file_df.groupby(['run_month', 'channel', 'key'], as_index = False, group_keys = False)['MoM P3M growth'].shift(0)\
                                .rolling(window=3, min_periods=3).apply(lambda x: np.sum(x >= 20), raw=True) 

In [254]:
trend_file_df['Avg(P3M Mean, Max)'] = trend_file_df[['P3M', 'P3M Max']].mean(axis=1)

In [255]:
trend_file_df.columns

Index(['channel', 'asm_area_code', 'brand_code', 'run_month', 'month_date',
       'extra_vol_co', 'other_co', 'price_off_co', 'seasonal_month_flag',
       'diwali', 'ganesh_chaturthi', 'drive', 'outlier', 'key', 'M month',
       'portfolio', 'qtr_ind_rate', 'sec_vol_actuals_rum_month', 'P3M', 'P6M',
       'LY P3M', 'LY P6M', 'LY P3M_copy', 'P3M Max', 'P3M Top 2 Mean',
       'MoM P3M growth', 'MoM P3M growth_lag_1', 'MoM P3M growth_lag_2',
       '>=20%_3M_inc_month_count', 'Avg(P3M Mean, Max)'],
      dtype='object')

In [256]:
for col in ['P3M', 'P6M', 'LY P3M', 'P3M Max', 'P3M Top 2 Mean', 
            'MoM P3M growth', '>=20%_3M_inc_month_count', 'Avg(P3M Mean, Max)',
            'MoM P3M growth_lag_1', 'MoM P3M growth_lag_2' ]:
    # if not 'LY' in col:  'LY P6M',
    trend_file_df.loc[trend_file_df['month_date'] > trend_file_df['run_month'], [col]] = np.nan
    trend_file_df[col] = trend_file_df.groupby(['key','channel'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )

In [257]:
trend_file_df

,channel,asm_area_code,brand_code,run_month,month_date,extra_vol_co,other_co,price_off_co,seasonal_month_flag,diwali,...,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)"
0,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-04-30,0.0,0.0,0.0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-05-31,0.0,0.0,0.0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00
2,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-06-30,0.0,0.0,0.0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00
3,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-07-31,0.0,0.0,0.0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00
4,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-08-31,0.0,0.0,0.0,0,0,...,NaN,NaN,NaN,0.00,0.00,NaN,NaN,NaN,NaN,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
214622,QCOM,QCW2,SW_SGPRF,2026-03-31,2026-11-30,0.0,0.0,0.0,0,1,...,49.28,4.52,0.64,6.96,6.96,68.965517,0.0,inf,0.0,9.36
214623,QCOM,QCW2,SW_SGPRF,2026-03-31,2026-12-31,0.0,0.0,0.0,0,0,...,49.28,3.84,0.00,6.96,6.96,68.965517,0.0,inf,0.0,9.36
214624,QCOM,QCW2,SW_SGPRF,2026-03-31,2027-01-31,0.0,0.0,0.0,0,0,...,49.28,7.00,6.96,6.96,6.96,68.965517,0.0,inf,0.0,9.36
214625,QCOM,QCW2,SW_SGPRF,2026-03-31,2027-02-28,0.0,0.0,0.0,0,0,...,49.28,3.80,6.96,6.96,6.96,68.965517,0.0,inf,0.0,9.36


In [258]:
for col in ['P3M', 'P6M', 'LY P3M', 'LY P6M']:
    trend_file_df[f'{col}_value'] = trend_file_df[col] * trend_file_df['qtr_ind_rate'] / (10 ** 7)

In [259]:
trend_file_df['sec_vol_actuals_rum_month_value'] = trend_file_df['qtr_ind_rate'] * trend_file_df['sec_vol_actuals_rum_month']/ (10 ** 7)
# trend_file_df['pred_prophet_70%ile_value'] = trend_file_df['qtr_ind_rate'] * trend_file_df['pred_prophet_70%ile']/ (10 ** 7)

In [260]:
value_cols = [col for col in trend_file_df.columns if 'value' in col]
value_cols

['P3M_value',
 'P6M_value',
 'LY P3M_value',
 'LY P6M_value',
 'sec_vol_actuals_rum_month_value']

In [261]:
for col in value_cols:
    try:
        assert trend_file_df[col].min() >= 0
    except:
        print(col)

    # trend_file_df[col] = trend_file_df[col] / (10 ** 7)

In [262]:
trend_file_df

,channel,asm_area_code,brand_code,run_month,month_date,extra_vol_co,other_co,price_off_co,seasonal_month_flag,diwali,...,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,sec_vol_actuals_rum_month_value
0,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-04-30,0.0,0.0,0.0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
1,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-05-31,0.0,0.0,0.0,0,0,...,NaN,NaN,NaN,NaN,0.00,0.000000,NaN,NaN,NaN,0.0
2,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-06-30,0.0,0.0,0.0,0,0,...,NaN,NaN,NaN,NaN,0.00,0.000000,NaN,NaN,NaN,0.0
3,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-07-31,0.0,0.0,0.0,0,0,...,NaN,NaN,NaN,NaN,0.00,0.000000,NaN,NaN,NaN,0.0
4,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-08-31,0.0,0.0,0.0,0,0,...,NaN,NaN,NaN,NaN,0.00,0.000000,NaN,NaN,NaN,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
214622,QCOM,QCW2,SW_SGPRF,2026-03-31,2026-11-30,0.0,0.0,0.0,0,1,...,68.965517,0.0,inf,0.0,9.36,0.001697,0.000849,0.007113,0.000652,0.0
214623,QCOM,QCW2,SW_SGPRF,2026-03-31,2026-12-31,0.0,0.0,0.0,0,0,...,68.965517,0.0,inf,0.0,9.36,0.001697,0.000849,0.007113,0.000554,0.0
214624,QCOM,QCW2,SW_SGPRF,2026-03-31,2027-01-31,0.0,0.0,0.0,0,0,...,68.965517,0.0,inf,0.0,9.36,0.001697,0.000849,0.007113,0.001010,0.0
214625,QCOM,QCW2,SW_SGPRF,2026-03-31,2027-02-28,0.0,0.0,0.0,0,0,...,68.965517,0.0,inf,0.0,9.36,0.001697,0.000849,0.007113,0.000548,0.0


In [263]:
assert trend_file_df.duplicated(
    subset=['run_month', 'brand_code', 'key', 'channel', 'month_date']
).sum() == 0

In [264]:
trend_file_df = trend_file_df.sort_values(
    ['channel', 'run_month', 'brand_code', 'key', 'month_date']
)
trend_file_df['LY'] = trend_file_df.groupby(
    ['channel', 'run_month', 'brand_code', 'key']
)['sec_vol_actuals_rum_month'].shift(12)

trend_file_df['LLY'] = trend_file_df.groupby(
    ['channel', 'run_month', 'brand_code', 'key']
)['sec_vol_actuals_rum_month'].shift(24)


trend_file_df['LY value'] = trend_file_df.groupby(
    ['channel', 'run_month', 'brand_code', 'key']
)['sec_vol_actuals_rum_month_value'].shift(12)

trend_file_df['LLY value'] = trend_file_df.groupby(
    ['channel', 'run_month', 'brand_code', 'key']
)['sec_vol_actuals_rum_month_value'].shift(24)


trend_file_df['Sec_Value_in_Cr_lag_1'] = trend_file_df.groupby(
    ['channel','run_month', 'brand_code', 'key']
)['sec_vol_actuals_rum_month_value'].shift(1)

trend_file_df['Sec_Value_in_Cr_lag_2'] = trend_file_df.groupby(
    ['channel','run_month', 'brand_code', 'key']
)['sec_vol_actuals_rum_month_value'].shift(2)

trend_file_df['Sec_Value_in_Cr_lag_3'] = trend_file_df.groupby(
    ['channel','run_month', 'brand_code', 'key']
)['sec_vol_actuals_rum_month_value'].shift(3)

In [265]:
for col in ['Sec_Value_in_Cr_lag_1', 'Sec_Value_in_Cr_lag_2', 'Sec_Value_in_Cr_lag_3']:
    # if not 'LY' in col:  'LY P6M',
    trend_file_df.loc[trend_file_df['month_date'] > trend_file_df['run_month'], [col]] = np.nan
    trend_file_df[col] = trend_file_df.groupby(['key','channel'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )

In [266]:
# trend_file_df[['ASM', 'Depot', 'PSKU']] = trend_file_df['key'].str.split('_', expand=True)

In [267]:
trend_file_df.reset_index(drop=True, inplace=True)

In [268]:
trend_file_df.shape

(214627, 42)

In [269]:
trend_file_df['key'].nunique()

4237

In [270]:
# brand_class_df = pd.read_excel(r"/data/aman_singh/acuuracy_check/Brand_Classification.xlsb")
# brand_class_df.head()
brand_class = pd.read_excel('/data/aman_singh/acuuracy_check/brand_class_new.xlsx')
brand_class['Channel'] = brand_class['Channel'].replace({
    'E-Commerce': 'ECOM', 'Q-Commerce': 'QCOM'})
brand_class.columns = brand_class.columns.str.lower()
brand_class = brand_class[['channel', 'brand','final class']]
brand_class.rename(columns = {'brand':'brand_code','final class':'class'}, inplace = True)


In [271]:
# brand_class_df.columns = ['brand_code', 'class']

In [272]:
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    brand_class, 
    on=['channel','brand_code'],
    how='left'
)
assert len_before_merge == len(trend_file_df)
del len_before_merge

In [273]:
# trend_file_df['final class'].isna().sum()

In [274]:
trend_file_df[trend_file_df['class'].isna()]['brand_code'].unique()

array(['JH_FRG_L', 'JH_HCR_KG', 'JH_HCR_L', 'JH_MKP_KG', 'JH_MKP_L',
       'JH_SCR_KG', 'JH_SCR_L', 'SFOATS-PR', 'LVN_SHMP', 'PA_GD_PLS',
       'P_GOHR_SR'], dtype=object)

In [275]:
trend_file_df['class'].unique()

array(['B', 'A', 'C', 'NPD', nan], dtype=object)

In [276]:
trend_file_df['key'].nunique()

4237

In [277]:
live_run_collated = pd.read_csv(collated_path)
final_missing_combinations = pd.DataFrame()

for channel in live_run_collated['channel'].unique():
    model_combinations = live_run_collated[live_run_collated['channel'] == channel]['key'].unique()

    channel_df = trend_file_df[trend_file_df['channel'] == channel]
    channel_df = channel_df[
        ~channel_df['key'].isin(model_combinations)
    ]

    final_missing_combinations = pd.concat([
        final_missing_combinations, channel_df
    ], ignore_index=False)

    del channel_df

In [278]:
live_run_collated['brand_code'].unique()

array(['ADV-AHO-R', 'BIO OILS', 'BRD_BDSPR', 'BRD_DOGAS', 'BRD_FSWSH',
       'BRD_HROIL', 'BRD_HRWAX', 'CO_SO_BL', 'CO_SO_PCP', 'CO_SO_VCN',
       'H&C', 'H&C_ALMND', 'HC SNS', 'HC_PBHOIL', 'KAYA_GM', 'KAYA_ML',
       'LIVON', 'LIVON S-R', 'LVNPST_ML', 'LVN_PRFSR', 'LVN_SRSNS',
       'LVN_SR_DR', 'MALO-NATU', 'MALT-NATU', 'NHR-SABDM', 'NHR-UTTAM',
       'NHR_AMLGD', 'NIHAR NHO', 'PA-ALO-HO', 'PA-BDYLOT', 'PABABY_CM',
       'PABABY_GM', 'PABABY_ML', 'PADV-HOT', 'PADV-HRCR', 'PADVJAS-R',
       'PADV_SMPN', 'PA_CN_HO', 'PA_EXT_ML', 'PA_HR_MSK', 'PA_JASGLD',
       'PA_ONI_HO', 'PCNO FLEX', 'PCNO(R)', 'PURSNS_ML', 'P_AL_GOLD',
       'P_EN_ALM', 'P_EN_ARG', 'P_EN_BGHB', 'P_EN_CRSH', 'P_EN_RSMR',
       'REV.LQDST', 'REV.ST.', 'REV_LQFRG', 'SAF-MUSLI', 'SAFF ACTV',
       'SAFF GOLD', 'SAFF KO', 'SAFF KOCO', 'SAFF OATS', 'SAFF SALT',
       'SAFF_ODLS', 'SAF_HONEY', 'SAF_MAYO', 'SAF_MILET', 'SAF_PNBTR',
       'SFOATS-FL', 'SFOATS_GD', 'SF_IM_CHY', 'SF_MNCHPS', 'SF_SOYACN',
       'S

In [279]:
final_missing_combinations['brand_code']

2674      CO_SO_PCP
2675      CO_SO_PCP
2676      CO_SO_PCP
2677      CO_SO_PCP
2678      CO_SO_PCP
            ...    
214453     SW_SGPRF
214454     SW_SGPRF
214455     SW_SGPRF
214456     SW_SGPRF
214457     SW_SGPRF
Name: brand_code, Length: 12471, dtype: object

In [280]:
# final_missing_combinations[final_missing_combinations['brand_code'] == 'SAF_HONEY']

In [281]:
# check = trend_file_df.groupby('key').agg(
#     size=('month_date', 'size'),
#     min_monthdate=('month_date', 'min'),
#     max_monthdate=('month_date', 'max')
# ).reset_index()


In [282]:
# check

In [283]:
# check['manual_dt_check'] = check.apply(
#     lambda x: len(pd.date_range(x['min_monthdate'], x['max_monthdate'], freq='M')),
#     axis=1
# )

In [284]:
# check[check['size'] != check['manual_dt_check']]

In [285]:
final_missing_combinations['month_date'].min()

Timestamp('2025-03-31 00:00:00')

In [300]:
final_missing_combinations.to_csv("/data/aman_singh/acuuracy_check/missing_df_all_brand_asm.csv", index=False)
#[final_missing_combinations['M month'].notna()]

In [ ]:
# import pandas as pd
# final_missing_combinations = pd.read_csv("/data/aman_singh/acuuracy_check/missing_df_all4.csv")

In [286]:
final_missing_combinations[final_missing_combinations['month_date'] == '2026-04-30']['P3M_value'].sum()

12.720115567509334

In [ ]:
# final_missing_combinations[final_missing_combinations['brand_code'] == 'SAF_HONEY'].to_csv("saf_honey2 Missing Forecasts jan'25.csv", index=False)

### basic checks

In [293]:
query = """select * from TRN_MIL_DF_ASM_BRAND
where run_month = '2026-03-31' """

data = pd.read_sql(con=dev_conn, sql=query)
data.columns = data.columns.str.lower()
data

,channel,asm_area_code,brand_code,run_month,month_date,sec_vol_actuals_rum_month,extra_vol_co,other_co,price_off_co,seasonal_month_flag,diwali,ganesh_chaturthi,drive,outlier
0,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-04-30,18.0,0.0,0.0,0.0,0,0,0,NaN,NaN
1,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-05-31,9.0,0.0,0.0,0.0,0,0,0,NaN,NaN
2,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-06-30,72.0,0.0,0.0,0.0,0,0,0,NaN,NaN
3,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-07-31,0.0,0.0,0.0,0.0,0,0,0,NaN,NaN
4,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-08-31,0.0,0.0,0.0,0.0,0,0,1,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
214622,QCOM,QCW2,SW_SGPRF,2026-03-31,2026-11-30,0.0,0.0,0.0,0.0,0,1,0,NaN,NaN
214623,QCOM,QCW2,SW_SGPRF,2026-03-31,2026-12-31,0.0,0.0,0.0,0.0,0,0,0,NaN,NaN
214624,QCOM,QCW2,SW_SGPRF,2026-03-31,2027-01-31,0.0,0.0,0.0,0.0,0,0,0,NaN,NaN
214625,QCOM,QCW2,SW_SGPRF,2026-03-31,2027-02-28,0.0,0.0,0.0,0.0,0,0,0,NaN,NaN


In [294]:
data['key'] = (
    data['asm_area_code'].astype(str) + '_' +
    data['brand_code'].astype(str)
    
)
data = data[data['key'].isin(final_missing_combinations['key'].unique())]

In [295]:
final_missing_combinations.groupby('key')['sec_vol_actuals_rum_month'].sum().reset_index().sort_values(by='sec_vol_actuals_rum_month', ascending=False)

,key,sec_vol_actuals_rum_month
648,WUP_NHR_ALOAM,261461.129
464,NUP_NHR_ALOAM,222029.079
109,CUP_NHR_ALOAM,105400.958
472,PJB_NHR_ALOAM,84192.775
306,HAR_NHR_ALOAM,71945.800
...,...,...
437,MCW1_PA_GD_PLS,0.008
102,CMB_SFOAT-CUP,0.008
162,ECE2_SAF_MAYO,0.006
107,CUP_CO_SO_VCN,0.006


In [120]:
final_missing_combinations[final_missing_combinations['key'] == 'MCW1_D354_731857'][['month_date', 'sec_vol_actuals_rum_month','P3M']]

,month_date,sec_vol_actuals_rum_month,P3M
724818,2025-10-31,2385.0,0.0
724819,2025-11-30,8190.0,1192.5
724820,2025-12-31,6675.0,5287.5
724821,2026-01-31,6876.0,5750.0
724822,2026-02-28,5865.0,7247.0
724823,2026-03-31,0.0,6472.0
724824,2026-04-30,0.0,6472.0
724825,2026-05-31,0.0,6472.0
724826,2026-06-30,0.0,6472.0
724827,2026-07-31,0.0,6472.0


In [296]:
import pandas as pd

as_of_date = pd.to_datetime("2026-03-31")  # month-end for Feb 2026
data['month_date'] = pd.to_datetime(data['month_date'])
filtered = data[
    
    (data['month_date'] < as_of_date) &
    (data['month_date'] >= as_of_date - pd.DateOffset(months=3))
]


In [297]:
filtered

,channel,asm_area_code,brand_code,run_month,month_date,sec_vol_actuals_rum_month,extra_vol_co,other_co,price_off_co,seasonal_month_flag,diwali,ganesh_chaturthi,drive,outlier,key
373,ECOM,BCE1,H&C_ALMND,2026-03-31,2025-12-31,30.000,0.0,0.0,0.0,0,0,0,NaN,NaN,BCE1_H&C_ALMND
374,ECOM,BCE1,H&C_ALMND,2026-03-31,2026-01-31,60.000,0.0,0.0,0.0,0,0,0,NaN,NaN,BCE1_H&C_ALMND
375,ECOM,BCE1,H&C_ALMND,2026-03-31,2026-02-28,135.000,0.0,0.0,0.0,0,0,0,NaN,NaN,BCE1_H&C_ALMND
753,ECOM,BCE1,NHR_ALOAM,2026-03-31,2025-12-31,0.000,0.0,0.0,0.0,0,0,0,NaN,NaN,BCE1_NHR_ALOAM
754,ECOM,BCE1,NHR_ALOAM,2026-03-31,2026-01-31,0.000,0.0,0.0,0.0,0,0,0,NaN,NaN,BCE1_NHR_ALOAM
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
213948,QCOM,QCW2,SAF_CDPRS,2026-03-31,2026-02-28,4.032,0.0,0.0,0.0,0,0,0,NaN,NaN,QCW2_SAF_CDPRS
214191,QCOM,QCW2,SFOATS-PR,2026-03-31,2026-02-28,0.120,0.0,0.0,0.0,0,0,0,NaN,NaN,QCW2_SFOATS-PR
214245,QCOM,QCW2,SFOATS_MG,2026-03-31,2025-12-31,0.014,0.0,0.0,0.0,0,0,0,NaN,NaN,QCW2_SFOATS_MG
214246,QCOM,QCW2,SFOATS_MG,2026-03-31,2026-01-31,0.044,0.0,0.0,0.0,0,0,0,NaN,NaN,QCW2_SFOATS_MG


In [298]:
# assert p3m equals
x = filtered.groupby(['month_date'])['sec_vol_actuals_rum_month'].sum().reset_index()['sec_vol_actuals_rum_month'].mean()
y = final_missing_combinations[final_missing_combinations['month_date'] == '2026-03-31']['P3M'].sum()
assert(int(x)==int(y))

AssertionError: 

In [299]:
(x,y)

(276275.6973333334, 298373.71716666216)

In [117]:
# p3m consistency check
as_of_date = pd.to_datetime('2026-03-31')

next_3_months = pd.date_range(
    start=as_of_date ,
    periods=4,
    freq='M'
)
for dt in next_3_months:
    p3m_sum = final_missing_combinations.loc[
        final_missing_combinations['month_date'] == dt, 'P3M'
    ].sum()
    
    print(f"P3M sum for {dt.date()}: {p3m_sum}")


P3M sum for 2026-03-31: 109378.89916666619
P3M sum for 2026-04-30: 109378.89916666619
P3M sum for 2026-05-31: 109378.89916666619
P3M sum for 2026-06-30: 109378.89916666619


### other work

In [7]:
trend_file_df = pd.read_sql(
    f"""
    select * from trn_mil_df_data
    where run_month='{run_month}'
""",
dev_conn
)
trend_file_df.columns = trend_file_df.columns.str.lower()
trend_file_df

,month_date,asm_area_code,depot_code,parent_material_code,sec_vol_actuals_rum_month,npd_flag,brand_code,qtr_ind_rate,free,extra_vol_co,...,npt_eve,print_spends_in_lacs,tv_spends_in_lacs,radio_spends_in_lacs,npt,pt,npd,update_timestamp,drive,outlier
0,2025-03-31,ECN2,D117,718553,0.0,0.0,PA-BDYLOT,216.8677,0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,NaN
1,2025-04-30,ECN2,D117,718553,96.0,0.0,PA-BDYLOT,216.8677,0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,NaN
2,2025-05-31,ECN2,D117,718553,105.6,0.0,PA-BDYLOT,216.8677,0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,NaN
3,2025-06-30,ECN2,D117,718553,105.6,0.0,PA-BDYLOT,216.8677,0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,NaN
4,2025-07-31,ECN2,D117,718553,220.8,0.0,PA-BDYLOT,216.8677,0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1233279,2024-10-31,ECN2,D117,718553,163.2,0.0,PA-BDYLOT,216.8677,0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,NaN
1233280,2024-11-30,ECN2,D117,718553,153.6,0.0,PA-BDYLOT,216.8677,0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,NaN
1233281,2024-12-31,ECN2,D117,718553,355.2,0.0,PA-BDYLOT,216.8677,0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,NaN
1233282,2025-01-31,ECN2,D117,718553,374.4,0.0,PA-BDYLOT,216.8677,0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,NaN


In [8]:
trend_file_df['channel'].unique()

array(['ECOM', 'GT', 'QCOM', 'MT'], dtype=object)

In [9]:
trend_file_df['run_month'].unique()

array(['2026-03-31'], dtype=object)

In [10]:
brand_asm_df = trend_file_df.groupby(['channel','asm_area_code','brand_code','run_month','month_date']).agg({'sec_vol_actuals_rum_month':'sum',
                        'extra_vol_co':'max', 'other_co':'max', 'price_off_co':'max',
                        'seasonal_month_flag':'max', 'diwali':'max', 'ganesh_chaturthi':'max',
                        'drive':'max', 'outlier':'max'}).reset_index()
brand_asm_df

,channel,asm_area_code,brand_code,run_month,month_date,sec_vol_actuals_rum_month,extra_vol_co,other_co,price_off_co,seasonal_month_flag,diwali,ganesh_chaturthi,drive,outlier
0,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-04-30,18.0,0.0,0.0,0.0,0,0,0,NaN,NaN
1,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-05-31,9.0,0.0,0.0,0.0,0,0,0,NaN,NaN
2,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-06-30,72.0,0.0,0.0,0.0,0,0,0,NaN,NaN
3,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-07-31,0.0,0.0,0.0,0.0,0,0,0,NaN,NaN
4,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-08-31,0.0,0.0,0.0,0.0,0,0,1,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
214622,QCOM,QCW2,SW_SGPRF,2026-03-31,2026-11-30,0.0,0.0,0.0,0.0,0,1,0,NaN,NaN
214623,QCOM,QCW2,SW_SGPRF,2026-03-31,2026-12-31,0.0,0.0,0.0,0.0,0,0,0,NaN,NaN
214624,QCOM,QCW2,SW_SGPRF,2026-03-31,2027-01-31,0.0,0.0,0.0,0.0,0,0,0,NaN,NaN
214625,QCOM,QCW2,SW_SGPRF,2026-03-31,2027-02-28,0.0,0.0,0.0,0.0,0,0,0,NaN,NaN


In [11]:
upload_df = brand_asm_df.copy()
upload_df['run_month'] = upload_df['run_month'].astype(str)
upload_df['month_date'] = upload_df['month_date'].astype(str)
upload_df.columns = upload_df.columns.str.upper()
upload_df

,CHANNEL,ASM_AREA_CODE,BRAND_CODE,RUN_MONTH,MONTH_DATE,SEC_VOL_ACTUALS_RUM_MONTH,EXTRA_VOL_CO,OTHER_CO,PRICE_OFF_CO,SEASONAL_MONTH_FLAG,DIWALI,GANESH_CHATURTHI,DRIVE,OUTLIER
0,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-04-30,18.0,0.0,0.0,0.0,0,0,0,NaN,NaN
1,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-05-31,9.0,0.0,0.0,0.0,0,0,0,NaN,NaN
2,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-06-30,72.0,0.0,0.0,0.0,0,0,0,NaN,NaN
3,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-07-31,0.0,0.0,0.0,0.0,0,0,0,NaN,NaN
4,ECOM,BCE1,ADV-AHO-R,2026-03-31,2022-08-31,0.0,0.0,0.0,0.0,0,0,1,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
214622,QCOM,QCW2,SW_SGPRF,2026-03-31,2026-11-30,0.0,0.0,0.0,0.0,0,1,0,NaN,NaN
214623,QCOM,QCW2,SW_SGPRF,2026-03-31,2026-12-31,0.0,0.0,0.0,0.0,0,0,0,NaN,NaN
214624,QCOM,QCW2,SW_SGPRF,2026-03-31,2027-01-31,0.0,0.0,0.0,0.0,0,0,0,NaN,NaN
214625,QCOM,QCW2,SW_SGPRF,2026-03-31,2027-02-28,0.0,0.0,0.0,0.0,0,0,0,NaN,NaN


In [12]:
from snowflake.connector.pandas_tools import write_pandas

write_pandas(dev_conn, upload_df, 
            table_name = "TRN_MIL_DF_ASM_BRAND",
            auto_create_table=True,
            overwrite = False,)

(True,
 1,
 214627,
 [('jvkedzpqmh/file0.txt',
   'LOADED',
   214627,
   214627,
   1,
   0,
   None,
   None,
   None,
   None)])

In [7]:
start_date = '2023-01-01'
end_date = '2026-02-28'
last_date = str(pd.to_datetime(end_date) - MonthEnd(1))
query = """
        SELECT
            CM.asm_area_code,
            CM.depot_code,
            MM.parent_material_code,
            LAST_DAY(MESR.month_date) AS month_date,
            SUM(MESR.sec_apo_plan_vol_rum) AS Sec_Vol_Apo_Plan_Rum_Month,
            SUM(MESR.sec_actuals_vol_rum) AS Sec_Vol_Actuals_Rum_Month
        FROM
            dwh_bpm_dist_sku_daily MESR
        JOIN
        (
            SELECT
                material_code,
                parent_material_code,
                material_group_code,
                uom_reporting,
                vol_per_unit
            FROM 
                mst_material
            WHERE
                company_code='MIL' AND
                latest_record_ind=1
                
        ) MM ON MESR.material_code = MM.material_code
        JOIN
        (
            SELECT DISTINCT
                channel_name, 
                asm_area_code,
                customer_code,
                depot_code
            FROM
                mst_customer
            WHERE
                company_code='MIL' AND
                latest_record_ind=1
                AND channel_name = '{}'
        ) CM ON MESR.distributor_code = CM.customer_code
        WHERE
            month_date BETWEEN '{}' and '{}'
            
        GROUP BY 1, 2, 3, 4
        ORDER BY 1, 2, 3, 4
        """.format(
        channel, start_date, last_date
    )

In [28]:
mesr_df = pd.read_sql(
    query,
prod_conn
)
#mesr_df.columns = mesr_df.columns.str.lower()
mesr_df

,ASM_AREA_CODE,DEPOT_CODE,PARENT_MATERIAL_CODE,MONTH_DATE,SEC_VOL_APO_PLAN_RUM_MONTH,SEC_VOL_ACTUALS_RUM_MONTH
0,AURG,D356,715095,2024-04-30,0.0,0.00
1,AURG,D356,715096,2024-04-30,0.0,0.00
2,AURG,D356,715098,2024-04-30,0.0,0.00
3,AURG,D356,715099,2024-04-30,0.0,0.00
4,AURG,D356,715100,2024-04-30,0.0,0.00
...,...,...,...,...,...,...
958817,None,SNQZ,725845,2025-05-31,0.0,0.00
958818,None,SNQZ,725845,2025-10-31,0.0,286.56
958819,None,SNQZ,725845,2025-12-31,0.0,792.00
958820,None,SNQZ,725845,2026-01-31,0.0,221.04


In [23]:
billwise_query = """ SELECT
        CM.asm_area_code,
        CM.depot_code,
        MM.parent_material_code,
        LAST_DAY(billwise.sales_invoice_date) AS month_date,
        SUM(billwise.sec_vol_actuals_rum) AS sec_actuals_vol_rum
    FROM PRD_DB.PUBLIC.DWH_RET_SKU_ACT_DAILY billwise
    JOIN (SELECT material_code, parent_material_code, material_group_code FROM PRD_DB.PUBLIC.MST_MATERIAL WHERE company_code='MIL' AND latest_record_ind=1 AND material_code >= '700000') MM ON billwise.material_code = MM.material_code
    JOIN (SELECT DISTINCT channel_name, asm_area_code, customer_code, depot_code FROM PRD_DB.PUBLIC.MST_CUSTOMER WHERE company_code='MIL' AND latest_record_ind=1 AND channel_name = '{}') CM ON billwise.distributor_code = CM.customer_code
    WHERE billwise.company_code = 'MIL' AND LAST_DAY(billwise.sales_invoice_date) = '{}'
    GROUP BY 1, 2, 3, 4 """.format(
        channel, end_date
    )

In [27]:
billwise_df = pd.read_sql(
    billwise_query,
prod_conn
)
#billwise_df.columns = billwise_df.columns.str.lower()
billwise_df

,ASM_AREA_CODE,DEPOT_CODE,PARENT_MATERIAL_CODE,MONTH_DATE,SEC_ACTUALS_VOL_RUM
0,DELM,D112,718322,2026-02-28,0.02000
1,BIHE,D232,718327,2026-02-28,0.00020
2,HYD1,D530,808271,2026-02-28,-0.00875
3,MUM2,D356,722330,2026-02-28,-0.00039
4,GUJS,D354,730808,2026-02-28,0.17600
...,...,...,...,...,...
8231,VIJ,D572,718849,2026-02-28,0.20000
8232,VIJ,D572,722229,2026-02-28,-0.20000
8233,BIHE,D233,724638,2026-02-28,-2.27500
8234,CUP,D113,718687,2026-02-28,1.00000


In [29]:
# billwise_df.duplicated(subset=['asm_area_code', 'depot_code', 'parent_material_code', 'month_date']).sum()
# billwise_df['sec_actuals_vol_rum'].sum()
billwise_df.rename(columns={'SEC_ACTUALS_VOL_RUM':'SEC_VOL_ACTUALS_RUM_MONTH'}, inplace=True)
billwise_df

,ASM_AREA_CODE,DEPOT_CODE,PARENT_MATERIAL_CODE,MONTH_DATE,SEC_VOL_ACTUALS_RUM_MONTH
0,DELM,D112,718322,2026-02-28,0.02000
1,BIHE,D232,718327,2026-02-28,0.00020
2,HYD1,D530,808271,2026-02-28,-0.00875
3,MUM2,D356,722330,2026-02-28,-0.00039
4,GUJS,D354,730808,2026-02-28,0.17600
...,...,...,...,...,...
8231,VIJ,D572,718849,2026-02-28,0.20000
8232,VIJ,D572,722229,2026-02-28,-0.20000
8233,BIHE,D233,724638,2026-02-28,-2.27500
8234,CUP,D113,718687,2026-02-28,1.00000


In [30]:
final_df = pd.concat(
    [mesr_df, billwise_df],
    ignore_index=True
)
final_df

,ASM_AREA_CODE,DEPOT_CODE,PARENT_MATERIAL_CODE,MONTH_DATE,SEC_VOL_APO_PLAN_RUM_MONTH,SEC_VOL_ACTUALS_RUM_MONTH
0,AURG,D356,715095,2024-04-30,0.0,0.000
1,AURG,D356,715096,2024-04-30,0.0,0.000
2,AURG,D356,715098,2024-04-30,0.0,0.000
3,AURG,D356,715099,2024-04-30,0.0,0.000
4,AURG,D356,715100,2024-04-30,0.0,0.000
...,...,...,...,...,...,...
967053,VIJ,D572,718849,2026-02-28,NaN,0.200
967054,VIJ,D572,722229,2026-02-28,NaN,-0.200
967055,BIHE,D233,724638,2026-02-28,NaN,-2.275
967056,CUP,D113,718687,2026-02-28,NaN,1.000
